In [ ]:
# Setup
import os, sys

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('main.ipynb'))
ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

DATA = os.path.join(ROOT, 'data', 'entradaProj2.25TAG.txt')
FIGS = os.path.join(ROOT, 'notebooks', 'figs')
RESULTS = os.path.join(ROOT, 'results')

In [ ]:
# Imports
import importlib
import pandas as pd
import src.parser
import src.utils
import src.allocator
import src.visualizer
importlib.reload(src.parser)
importlib.reload(src.utils)
importlib.reload(src.allocator)
importlib.reload(src.visualizer)
from src.parser import load_input
from src.utils import build_project_prefs, rank_in_project, rank_in_student, is_stable, student_satisfaction, project_satisfaction, ensure_non_empty_projects
from src.allocator import run_gale_shapley
from src.visualizer import visualize_iteration

In [ ]:
# Execução do algoritmo e geração das visualizações
projects, students = load_input(DATA)
matching, logs = run_gale_shapley(students, projects, max_iter=1000, fixed_iterations=10)

print(f"Total de alunos: {len(students)}")
print(f"Total de projetos: {len(projects)}")
print(f"Total de vagas: {sum(p.vacancies for p in projects.values())}\n")

for i in range(1, 11):
    state = logs.get(i, {'proposals': [], 'accepted': [], 'rejected': []})
    print(f"Iteração {i}: {len(state['proposals'])} propostas, {len(state['accepted'])} aceitos, {len(state['rejected'])} rejeitados")
    out = os.path.join(FIGS, f'iter_{i:02d}.png')
    visualize_iteration(students, projects, state, i, out)

print(f"\nAlunos emparelhados: {sum(1 for v in matching.values() if v is not None)}")

Visualização das 10 iterações do algoritmo Gale-Shapley:
Total de alunos: 200
Total de projetos: 50
Total de vagas: 80

Iteração 1: 200 propostas, 43 aceitos temporariamente, 157 rejeitados
Iteração 2: 157 propostas, 52 aceitos temporariamente, 148 rejeitados
Iteração 2: 157 propostas, 52 aceitos temporariamente, 148 rejeitados
Iteração 3: 144 propostas, 58 aceitos temporariamente, 138 rejeitados
Iteração 3: 144 propostas, 58 aceitos temporariamente, 138 rejeitados
Iteração 4: 9 propostas, 58 aceitos temporariamente, 9 rejeitados
Iteração 4: 9 propostas, 58 aceitos temporariamente, 9 rejeitados
Iteração 5: 3 propostas, 58 aceitos temporariamente, 3 rejeitados
Iteração 5: 3 propostas, 58 aceitos temporariamente, 3 rejeitados
Iteração 6: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
Iteração 6: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
Iteração 7: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
Iteração 7: 0 propostas, 58 aceitos temporariamente, 0 rejeitados
It

In [ ]:
# Pós-processamento e matriz de emparelhamento
print(f"Projetos vazios antes: {sum(1 for p in projects.values() if not p.current_alloc)}")
still_empty = ensure_non_empty_projects(projects, students, matching)
print(f"Projetos vazios depois: {still_empty}")

# Matriz de emparelhamento
rows = []
for sid, proj_code in sorted(matching.items()):
    if proj_code is None: 
        continue
    proj = projects[proj_code]
    stu = students[sid]
    proj_rank = rank_in_project(proj, sid)
    stu_rank = rank_in_student(stu, proj_code)
    
    rows.append({
        'Aluno': f'A{sid}',
        'Projeto': proj_code,
        'Rank Aluno (Projeto)': f'{proj_rank}º' if proj_rank < 10**9 else 'N/A',
        'Rank Projeto (Aluno)': f'{stu_rank}ª' if stu_rank <= 3 else 'N/A',
    })

df = pd.DataFrame(rows)
os.makedirs(RESULTS, exist_ok=True)
df.to_csv(os.path.join(RESULTS, 'final_matching.csv'), index=False)

print(f"\n{df.to_string(index=False)}")

# Métricas
print(f"\nEstável: {is_stable(matching, projects, students)}")
sat = student_satisfaction(matching, students)
print(f"Satisfação: 1ª={sat['choice1_pct']:.1f}%, 2ª={sat['choice2_pct']:.1f}%, 3ª={sat['choice3_pct']:.1f}%")
print(f"Rank médio (projetos): {project_satisfaction(projects):.2f}")

=== Antes do pós-processamento ===
  Alunos emparelhados: 58
  Projetos vazios: 11

=== Análise de projetos vazios ===
  P13: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P19: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P23: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P31: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P32: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P33: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P42: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P44: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P46: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P48: min_req=5, elegíveis=30, não-emparelhados elegíveis=0
  P50: min_req=5, elegíveis=30, não-emparelhados elegíveis=0

=== Após pós-processamento ===
  Alunos emparelhados: 58
  Projetos ainda vazios: 1
  NOTA: 1 projetos permanecem vazios pois não há alunos elegíveis suficientes (min_req > notas disponíveis)

=== Matriz de Em

In [ ]:
# Resumo
print(f"Alunos: {len(students)}")
print(f"Projetos: {len(projects)}")
print(f"Emparelhados: {sum(1 for v in matching.values() if v is not None)}")
print(f"Projetos preenchidos: {sum(1 for p in projects.values() if p.current_alloc)}")

empty = [p for p in projects.values() if not p.current_alloc]
if empty:
    print(f"\nProjetos vazios ({len(empty)}):")
    for p in empty:
        print(f"  {p.code} (min_req={p.min_req})")

print(f"\nPrimeiros 15 emparelhamentos:")
print(df.head(15).to_string(index=False))

=== RESUMO FINAL ===
Total de alunos: 200
Total de projetos: 50
Alunos emparelhados: 58
Projetos com pelo menos 1 aluno: 49
Projetos vazios: 1

Projetos que permaneceram vazios:
  P50: min_req=5, alunos elegíveis no total=30

NOTA: Impossível preencher todos os projetos - há mais vagas exigindo nota 5 do que alunos com nota 5.
  Alunos com nota 5: 30
  Vagas que exigem nota 5: 35

=== Primeiros 15 emparelhamentos (exemplo conforme especificação) ===
Aluno Projeto Emparelhado Rank do Aluno (Lista do Projeto) Rank do Projeto (Lista do Aluno)
   A1                 P32                              N/A                      Não listado
   A2                  P1              2º (de 5 elegíveis)                       1ª escolha
   A6                  P9              1º (de 8 elegíveis)                       2ª escolha
   A8                  P6              1º (de 1 elegíveis)                       1ª escolha
  A10                 P43              1º (de 9 elegíveis)                       2ª es

## Exportar PDF
```
jupyter nbconvert --to pdf notebooks/main.ipynb --output mini-relatorio.pdf
```